# Examen Final – Sistema de Recuperación de Información con RAG
## ICCD753 Recuperación de Información 2026-A
### Corpus: arXiv Paper Abstracts

**Estudiante:** Francisco Hernández  
**Fecha:** 2026-07-15

---

## Descripción del Sistema

Este notebook implementa un **Sistema de Recuperación Aumentada por Generación (RAG)** completo sobre el corpus de resúmenes de artículos científicos de arXiv. El pipeline consta de:

1. Preparación y preprocesamiento del corpus
2. Representación mediante embeddings semánticos
3. Almacenamiento y búsqueda vectorial con FAISS
4. Recuperación inicial (First-Stage Retrieval)
5. Generación de respuestas con LLM (Google Gemini)
6. Presentación de evidencias recuperadas
7. Interfaz web conversacional (Next.js en AWS Amplify)
8. Despliegue en la nube (EC2 t2.micro + Amplify)
9. Evaluación subjetiva del sistema

**URL del sistema desplegado:** `http://<EC2_PUBLIC_IP>:8000` (backend) | `https://<amplify-app>.amplifyapp.com` (frontend chat)

# Sección A: Preparación del Corpus

Se utiliza el dataset **arXiv Paper Abstracts** disponible en Kaggle (`arxiv_data.csv`), que contiene títulos, resúmenes y tópicos de artículos científicos. Para garantizar tiempos de respuesta razonables en hardware de CPU, se toma una muestra representativa de **50,000 documentos**.

El campo `text` es la concatenación de título y abstract, que es la unidad de recuperación del sistema.

In [ ]:
import pandas as pd
import numpy as np
import re
import os

# ── Configuración ──────────────────────────────────────────────────────────────
DATA_PATH = "archive (1)/arxiv_data.csv"   # Ruta local al CSV del corpus
NUM_SAMPLES = 50_000                        # Documentos a indexar
TOP_K_INITIAL = 50                          # Candidatos para la primera etapa
TOP_K_RERANK  = 10                          # Resultados finales tras re-ranking
GEMINI_MODEL = "gemini-3.1-flash-lite"       # Modelo LLM para generación

# ── Carga del corpus ────────────────────────────────────────────────────────────
print(f"Cargando {NUM_SAMPLES:,} documentos del corpus arXiv...")
df_raw = pd.read_csv(DATA_PATH, nrows=NUM_SAMPLES)
df_raw.dropna(subset=["titles", "summaries"], inplace=True)
df_raw.reset_index(drop=True, inplace=True)

# Crear DataFrame limpio
df_docs = pd.DataFrame({
    "doc_id":   df_raw.index.astype(str),
    "title":    df_raw["titles"].str.strip(),
    "abstract": df_raw["summaries"].str.strip(),
    "terms":    df_raw["terms"].fillna(""),
    "text":     df_raw["titles"].str.strip() + ". " + df_raw["summaries"].str.strip()
})

print(f"✅ Documentos cargados: {len(df_docs):,}")
print(f"   Columnas: {list(df_docs.columns)}")
df_docs.head(3)

Cargando 50,000 documentos del corpus arXiv...
✅ Documentos cargados: 50,000
   Columnas: ['doc_id', 'title', 'abstract', 'terms', 'text']


,doc_id,title,abstract,terms,text
0,0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"['cs.CV', 'cs.LG']",Survey on Semantic Stereo Matching / Semantic ...
1,1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"['cs.CV', 'cs.AI', 'cs.LG']",FUTURE-AI: Guiding Principles and Consensus Re...
2,2,Enforcing Mutual Consistency of Hard Regions f...,"In this paper, we proposed a novel mutual cons...","['cs.CV', 'cs.AI']",Enforcing Mutual Consistency of Hard Regions f...


## Preprocesamiento de texto

Se aplica un pipeline de normalización básico:
- Conversión a minúsculas
- Eliminación de tokens no alfabéticos
- Eliminación de stopwords en inglés (NLTK)
- Stemming con Snowball Stemmer

El texto preprocesado (`clean_text`) se usa exclusivamente para los embeddings.

In [4]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

stop_words = set(stopwords.words('english'))
stemmer    = SnowballStemmer('english')

def preprocess(text: str) -> str:
    """Normaliza, tokeniza, elimina stopwords y aplica stemming."""
    text   = str(text).lower()
    tokens = re.findall(r'\b[a-z]+\b', text)
    clean  = [stemmer.stem(t) for t in tokens if t not in stop_words]
    return " ".join(clean)

print("Preprocesando textos del corpus...")
df_docs['clean_text'] = df_docs['text'].apply(preprocess)
print("Preprocesamiento completado.")
print(f"   Ejemplo → '{df_docs['clean_text'].iloc[0][:120]}...'")

[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1082)>
[nltk_data] Error loading punkt: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1082)>


Preprocesando textos del corpus...
Preprocesamiento completado.
   Ejemplo → 'survey semant stereo match semant depth estim stereo match one wide use techniqu infer depth stereo imag owe robust spee...'


# Sección B: Representación mediante Embeddings

Se usa el modelo `all-MiniLM-L6-v2` de `sentence-transformers` para generar embeddings semánticos densos. Este modelo produce vectores de **384 dimensiones** y está optimizado para recuperación de información (entrenado con pares pregunta-respuesta de MS-MARCO).

**Justificación:** Captura similitud semántica más allá de la coincidencia léxica, permitiendo recuperar documentos relevantes aunque no compartan las mismas palabras clave que la consulta.

In [5]:
from sentence_transformers import SentenceTransformer

# Cargar modelo de embeddings
print("Cargando modelo de embeddings 'all-MiniLM-L6-v2'...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generar embeddings para todos los documentos del corpus
print(f"Generando embeddings para {len(df_docs):,} documentos...")
doc_embeddings = embedding_model.encode(
    df_docs['clean_text'].tolist(),
    show_progress_bar=True,
    batch_size=256
)

print(f"\nEmbeddings generados.")
print(f"   Shape: {doc_embeddings.shape}")
print(f"   Tipo:  {doc_embeddings.dtype}")
print(f"   Dimensión por vector: {doc_embeddings.shape[1]}")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando modelo de embeddings 'all-MiniLM-L6-v2'...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10140.68it/s]


Generando embeddings para 50,000 documentos...


Batches: 100%|██████████| 196/196 [01:44<00:00,  1.87it/s]


Embeddings generados.
   Shape: (50000, 384)
   Tipo:  float32
   Dimensión por vector: 384


# Sección C: Almacenamiento y Búsqueda Vectorial (FAISS)

Se construye un índice **FAISS `IndexFlatL2`** (búsqueda exacta por distancia Euclidiana) para almacenar y consultar los vectores de documentos de forma eficiente.

**FAISS** (Facebook AI Similarity Search) permite búsquedas de vecinos más cercanos sobre millones de vectores en milisegundos. Para este corpus de 50K documentos, el índice plano garantiza resultados exactos sin comprometer la velocidad.

In [6]:
import faiss


# Construir índice FAISS plano (búsqueda exacta)
vector_dim = doc_embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(vector_dim)

# Añadir vectores al índice (requiere float32)
faiss_index.add(doc_embeddings.astype(np.float32))

print(f"Índice FAISS creado correctamente.")
print(f"   Vectores indexados: {faiss_index.ntotal:,}")
print(f"   Dimensión:          {vector_dim}")
print(f"   Tipo de índice:     IndexFlatL2 (búsqueda exacta)")

# Guardar índice y DataFrame para el backend
faiss.write_index(faiss_index, "arxiv_faiss.index")
df_docs.to_pickle("arxiv_docs.pkl")
print("\nÍndice y documentos guardados en disco:")
print("   - arxiv_faiss.index")
print("   - arxiv_docs.pkl")

Índice FAISS creado correctamente.
   Vectores indexados: 50,000
   Dimensión:          384
   Tipo de índice:     IndexFlatL2 (búsqueda exacta)

Índice y documentos guardados en disco:
   - arxiv_faiss.index
   - arxiv_docs.pkl


# Sección D: Recuperación (First-Stage Retrieval)

La recuperación inicial convierte la consulta del usuario en un embedding usando el **mismo modelo** que los documentos, y luego busca en el índice FAISS los `TOP_K_INITIAL` vectores más cercanos.

Esta etapa es muy rápida (~5ms para 50K docs) pero puede recuperar documentos temáticamente relacionados que no son óptimamente relevantes. Por eso existe el re-ranking en el siguiente paso.

In [7]:
def search_initial(query: str, k: int = TOP_K_INITIAL) -> list[dict]:
    """
    Primera etapa de recuperación: Embedding + búsqueda FAISS.
    
    Args:
        query: Consulta en lenguaje natural.
        k: Número de candidatos a recuperar.
    
    Returns:
        Lista de documentos candidatos con sus metadatos.
    """
    query_clean = preprocess(query)
    query_emb   = embedding_model.encode([query_clean]).astype(np.float32)
    
    distances, indices = faiss_index.search(query_emb, k)
    
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if 0 <= idx < len(df_docs):
            doc = df_docs.iloc[idx].to_dict()
            doc['l2_distance'] = float(dist)
            results.append(doc)
    return results

# ── Prueba de la función de recuperación ────────────────────────────────────────
test_query   = "deep learning applications in medical imaging"
test_results = search_initial(test_query, k=5)

print(f"Consulta: '{test_query}'")
print(f"Top-5 resultados iniciales:")
print("-" * 80)
for i, doc in enumerate(test_results):
    print(f"{i+1}. {doc['title'][:70]}")
    print(f"   L2 distance: {doc['l2_distance']:.4f}")

Consulta: 'deep learning applications in medical imaging'
Top-5 resultados iniciales:
--------------------------------------------------------------------------------
1. Deep Learning for Medical Image Processing: Overview, Challenges and F
   L2 distance: 0.6309
2. Deep Learning for Medical Image Processing: Overview, Challenges and F
   L2 distance: 0.6309
3. A Survey on Deep Learning in Medical Image Analysis
   L2 distance: 0.7003
4. Medical Image Analysis using Convolutional Neural Networks: A Review
   L2 distance: 0.7127
5. Explainable deep learning models in medical image analysis
   L2 distance: 0.7725


## Re-ranking con Cross-Encoder

El **Cross-Encoder** (`ms-marco-MiniLM-L-6-v2`) evalúa la relevancia de cada par `(consulta, documento)` de forma conjunta, produciendo un score más preciso que la similitud de embeddings. Se aplica sobre los `TOP_K_INITIAL` candidatos para seleccionar los `TOP_K_RERANK` finales.

In [8]:
from sentence_transformers import CrossEncoder

print("Cargando Cross-Encoder para re-ranking...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("Cross-Encoder listo.")

def rerank_results(query: str, candidates: list[dict], top_k: int = TOP_K_RERANK) -> list[dict]:
    """
    Segunda etapa: Re-ranking con Cross-Encoder.
    
    Args:
        query:      Consulta original del usuario.
        candidates: Lista de documentos candidatos (output de search_initial).
        top_k:      Número de resultados finales a retornar.
    
    Returns:
        Lista de documentos re-rankeados, ordenados por relevancia descendente.
    """
    if not candidates:
        return []
    
    # Crear pares (query, document_text) para el Cross-Encoder
    pairs  = [[query, doc['text'][:512]] for doc in candidates]  # Truncar a 512 chars
    scores = cross_encoder.predict(pairs)
    
    # Asignar scores y ordenar
    for doc, score in zip(candidates, scores):
        doc['rerank_score'] = float(score)
    
    ranked = sorted(candidates, key=lambda x: x['rerank_score'], reverse=True)
    return ranked[:top_k]

# ── Prueba del pipeline completo (D) ────────────────────────────────────────────
test_query      = "deep learning applications in medical imaging"
initial_results = search_initial(test_query, k=TOP_K_INITIAL)
final_results   = rerank_results(test_query, initial_results)

print(f"\nConsulta: '{test_query}'")
print(f"Top-{TOP_K_RERANK} resultados tras re-ranking:")
print("-" * 80)
for i, doc in enumerate(final_results):
    print(f"{i+1}. {doc['title'][:65]}")
    print(f"   Score re-ranking: {doc['rerank_score']:.4f}")

Cargando Cross-Encoder para re-ranking...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 14516.99it/s]


Cross-Encoder listo.

Consulta: 'deep learning applications in medical imaging'
Top-10 resultados tras re-ranking:
--------------------------------------------------------------------------------
1. Deep learning and its application to medical image segmentation
   Score re-ranking: 9.2160
2. Deep learning and its application to medical image segmentation
   Score re-ranking: 9.2160
3. Unsupervised domain adaptation for medical imaging segmentation w
   Score re-ranking: 7.2030
4. Unsupervised domain adaptation for medical imaging segmentation w
   Score re-ranking: 7.2030
5. Image Synthesis for Data Augmentation in Medical CT using Deep Re
   Score re-ranking: 7.1027
6. A Survey on Deep Learning in Medical Image Analysis
   Score re-ranking: 6.8309
7. An overview of deep learning in medical imaging focusing on MRI
   Score re-ranking: 6.7341
8. An overview of deep learning in medical imaging focusing on MRI
   Score re-ranking: 6.7341
9. Deep Learning Under the Microscope: Improving t

# Sección E: Generación Aumentada por Recuperación (RAG)

En esta etapa, los documentos recuperados y re-rankeados se usan como **contexto** para un Modelo de Lenguaje Grande (LLM). El LLM genera una respuesta fundamentada en el corpus, combinando la información de los top-K documentos.

**Modelo LLM:** Google Gemini 1.5 Flash  
**Estrategia de prompting:**
- Se incluyen los top-K documentos como contexto numerado
- Se instruye al modelo para responder únicamente basándose en el contexto
- Si el contexto es insuficiente, el modelo debe indicarlo explícitamente
- La respuesta incluye referencias a los documentos usados

In [14]:
#››%pip install google-generativeai sentence-transformers faiss-cpu pandas pyarrow nltk numpy
import google.generativeai as genai

# Configurar API de Gemini
# La API key se lee de variable de entorno (NO hardcodear en el código)
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
if not GEMINI_API_KEY:
    raise ValueError("⚠️  Configura la variable de entorno GEMINI_API_KEY antes de ejecutar.")

genai.configure(api_key=GEMINI_API_KEY)
llm = genai.GenerativeModel(GEMINI_MODEL)
GEMINI_MODEL = "gemini-3.1-flash-lite"
print(f" Modelo LLM configurado: {GEMINI_MODEL}")


# ── Plantilla del prompt RAG ────────────────────────────────────────────────────
RAG_SYSTEM_PROMPT = """Eres un asistente especializado en investigación científica. 
Tu tarea es responder preguntas basándote EXCLUSIVAMENTE en los fragmentos de papers científicos que se te proporcionan.
Reglas:
- Usa solo la información del contexto para responder.
- Si el contexto no contiene información suficiente, indícalo explícitamente.
- Cita los papers relevantes usando su número (e.g., [1], [2]).
- Responde en español si la pregunta está en español, en inglés si está en inglés.
- Sé conciso pero completo."""
def generate_rag_answer(query: str, reranked_docs: list[dict]) -> dict:
    if not reranked_docs:
        return {
            "answer": "No se encontraron documentos relevantes en el corpus para responder esta consulta.",
            "evidences": []
        }
    
    # Construir el bloque de contexto
    context_parts = []
    for i, doc in enumerate(reranked_docs[:5], start=1):  # Usar top-5 como contexto
        snippet = doc['abstract'][:400].strip().replace('\n', ' ')
        context_parts.append(f"[{i}] Título: {doc['title']}\nResumen: {snippet}")
    
    context_block = "\n\n".join(context_parts)
    
    # Construir el prompt completo
    user_prompt = f"""{RAG_SYSTEM_PROMPT}
--- CONTEXTO (Papers científicos recuperados) ---
{context_block}
--- FIN DEL CONTEXTO ---
Pregunta del usuario: {query}
Respuesta:"""
    
    # Llamar al LLM
    response = llm.generate_content(user_prompt)
    answer   = response.text.strip()
    
    # Preparar evidencias para el frontend
    evidences = [
        {
            "product_id":       doc['doc_id'],
            "title":            doc['title'],
            "abstract_snippet": doc['abstract'][:200] + "...",
            "image_url":        "",
            "similarity":       max(0.0, 1.0 / (1.0 + doc.get('l2_distance', 1.0)))
        }
        for doc in reranked_docs[:5]
    ]
    
    return {"answer": answer, "evidences": evidences}
# ── Prueba del pipeline RAG completo ─────────────────────────────────────────────
test_query   = "What are the main applications of Graph Neural Networks?"
initial      = search_initial(test_query, k=TOP_K_INITIAL)
reranked     = rerank_results(test_query, initial, top_k=TOP_K_RERANK)
# 2. Llamada corregida sin la "s" de más
rag_output   = generate_rag_answer(test_query, reranked)
print(f"\nConsulta: {test_query}")
print("=" * 80)
print("📝 Respuesta RAG:")
print(rag_output['answer'])
print("\nEvidencias utilizadas:")
for ev in rag_output['evidences']:
    print(f"  - [{ev['product_id']}] {ev['title'][:70]} (sim: {ev['similarity']:.3f})")

 Modelo LLM configurado: gemini-3.1-flash-lite

Consulta: What are the main applications of Graph Neural Networks?
📝 Respuesta RAG:
Las redes neuronales de grafos (GNNs) tienen diversas aplicaciones en múltiples dominios, entre los que se incluyen:

*   **Bioinformática y quimioinformática:** [1]
*   **Redes sociales:** [1]
*   **Procesamiento de lenguaje natural:** [1]
*   **Visión por computadora:** [1]
*   **Descubrimiento de fármacos:** [4]
*   **Sistemas de recomendación:** [4]
*   **Localización de redes (problemas de regresión no lineal a gran escala):** [5]

Adicionalmente, se utilizan de forma generalizada para la clasificación de datos estructurados en el contexto del aprendizaje automático [5].

Evidencias utilizadas:
  - [20322] Graph Capsule Convolutional Neural Networks (sim: 0.578)
  - [23136] AutoGraph: Automated Graph Neural Network (sim: 0.570)
  - [40300] AutoGraph: Automated Graph Neural Network (sim: 0.570)
  - [21319] Meta-Learning with Graph Neural Networks: Meth

# Sección F: Presentación de Evidencias

Cada respuesta del sistema RAG incluye las **evidencias** utilizadas para generar la respuesta. Esto permite al usuario:
1. Verificar la relación entre la consulta, los documentos recuperados y la respuesta generada
2. Acceder a los papers originales si desea profundizar
3. Evaluar si el LLM ha usado correctamente la información del corpus

Las evidencias muestran: **título del paper**, **snippet del abstract** y **score de similitud**.

In [16]:
from IPython.display import Markdown, display

def display_evidences(query: str, rag_result: dict) -> None:
    """
    Formatea y muestra las evidencias RAG en el notebook con Markdown.
    
    Args:
        query:      Consulta realizada por el usuario.
        rag_result: Resultado del pipeline RAG (answer + evidences).
    """
    md_lines = []
    md_lines.append(f"## Consulta\n> {query}")
    md_lines.append(f"\n## Respuesta Generada\n{rag_result['answer']}")
    md_lines.append("\n---\n## Evidencias Recuperadas")
    
    for i, ev in enumerate(rag_result['evidences'], start=1):
        sim_pct = ev['similarity'] * 100
        md_lines.append(
            f"\n### [{i}] {ev['title']}\n"
            f"- **ID:** `{ev['product_id']}`  \n"
            f"- **Similitud:** {sim_pct:.1f}%  \n"
            f"- **Abstract:** {ev['abstract_snippet']}"
        )
    
    display(Markdown("\n".join(md_lines)))

# ── Demo: 3 consultas representativas con sus evidencias ───────────────────────
test_queries = [
    "What are the main applications of Graph Neural Networks?",
    "How is reinforcement learning used in robotics?",
    "Recent advances in diffusion models for image generation."
]

import time  # Agrega esta línea arriba

for query in test_queries:
    initial  = search_initial(query, k=TOP_K_INITIAL)
    reranked = rerank_results(query, initial, top_k=TOP_K_RERANK)
    result   = generate_rag_answer(query, reranked)
    display_evidences(query, result)
    print("\n" + "=" * 100 + "\n")
    time.sleep(3)  # Pausa de 3 segundos para no saturar la API

## Consulta
> What are the main applications of Graph Neural Networks?

## Respuesta Generada
Las Redes Neuronales de Grafos (GNN, por sus siglas en inglés) se utilizan en una amplia variedad de dominios. Sus aplicaciones principales incluyen:

*   **Bioinformática y quimioinformática:** [1]
*   **Redes sociales:** [1]
*   **Procesamiento de lenguaje natural:** [1]
*   **Visión artificial:** [1]
*   **Descubrimiento de fármacos:** [4]
*   **Sistemas de recomendación:** [4]
*   **Localización de redes (problemas de regresión no lineal a gran escala):** [5]

Además, las GNN son empleadas para tareas de clasificación de datos estructurados en el contexto del aprendizaje automático [5] y análisis de grafos en general [2, 3].

---
## Evidencias Recuperadas

### [1] Graph Capsule Convolutional Neural Networks
- **ID:** `20322`  
- **Similitud:** 57.8%  
- **Abstract:** Graph Convolutional Neural Networks (GCNNs) are the most recent exciting
advancement in deep learning field and their applications are quickly spreading
in multi-cross-domains including bioinformatics...

### [2] AutoGraph: Automated Graph Neural Network
- **ID:** `23136`  
- **Similitud:** 57.0%  
- **Abstract:** Graphs play an important role in many applications. Recently, Graph Neural
Networks (GNNs) have achieved promising results in graph analysis tasks. Some
state-of-the-art GNN models have been proposed,...

### [3] AutoGraph: Automated Graph Neural Network
- **ID:** `40300`  
- **Similitud:** 57.0%  
- **Abstract:** Graphs play an important role in many applications. Recently, Graph Neural
Networks (GNNs) have achieved promising results in graph analysis tasks. Some
state-of-the-art GNN models have been proposed,...

### [4] Meta-Learning with Graph Neural Networks: Methods and Applications
- **ID:** `21319`  
- **Similitud:** 58.7%  
- **Abstract:** Graph Neural Networks (GNNs), a generalization of deep neural networks on
graph data have been widely used in various domains, ranging from drug
discovery to recommender systems. However, GNNs on such...

### [5] Graph Neural Network for Large-Scale Network Localization
- **ID:** `22639`  
- **Similitud:** 58.8%  
- **Abstract:** Graph neural networks (GNNs) are popular to use for classifying structured
data in the context of machine learning. But surprisingly, they are rarely
applied to regression problems. In this work, we a...

## Consulta
> How is reinforcement learning used in robotics?

## Respuesta Generada
El aprendizaje por refuerzo (*reinforcement learning* o RL) se utiliza en robótica para desarrollar soluciones adaptativas frente a tareas complejas y diversas que serían difíciles de programar manualmente [3]. Sus aplicaciones y enfoques incluyen:

*   **Control continuo:** El aprendizaje por refuerzo profundo permite que los algoritmos aprendan comportamientos complejos, manejen espacios de acción continuos y optimicen estrategias en entornos de alta dimensionalidad [1].
*   **Mejora de la eficiencia:** Se emplean métodos basados en modelos (como PILCO o BlackDrops) para mejorar la eficiencia de datos, o métodos libres de modelos (como PPO o Q-learning) para la aproximación de controladores óptimos [2].
*   **Aprendizaje secuencial:** Se investiga el aprendizaje por refuerzo en contextos de tareas múltiples donde el robot adquiere diversas habilidades de manera secuencial, adaptándose al entorno y a las necesidades del usuario [5].
*   **Resolución de tareas de manipulación:** Para superar la dificultad de aprender con señales de recompensa dispersas, se utilizan demostraciones (como las de nivel de objeto o de locomoción simulada) para hacer el proceso de aprendizaje más eficiente en términos de muestras [4].

A pesar de su potencial, su aplicación en robots reales presenta retos debido a la falta de guías estandarizadas, el consumo de tiempo y preocupaciones sobre la seguridad durante la interacción con el entorno [2, 3].

---
## Evidencias Recuperadas

### [1] Using Deep Reinforcement Learning for the Continuous Control of Robotic Arms
- **ID:** `35446`  
- **Similitud:** 64.1%  
- **Abstract:** Deep reinforcement learning enables algorithms to learn complex behavior,
deal with continuous action spaces and find good strategies in environments
with high dimensional state spaces. With deep rein...

### [2] Reinforcement Learning for Robotics and Control with Active Uncertainty Reduction
- **ID:** `34951`  
- **Similitud:** 57.9%  
- **Abstract:** Model-free reinforcement learning based methods such as Proximal Policy
Optimization, or Q-learning typically require thousands of interactions with
the environment to approximate the optimum controll...

### [3] Setting up a Reinforcement Learning Task with a Real-World Robot
- **ID:** `35775`  
- **Similitud:** 61.5%  
- **Abstract:** Reinforcement learning is a promising approach to developing hard-to-engineer
adaptive solutions for complex and diverse robotic tasks. However, learning
with real-world robots is often unreliable and...

### [4] Reinforcement Learning for Robotic Manipulation using Simulated Locomotion Demonstrations
- **ID:** `33464`  
- **Similitud:** 59.0%  
- **Abstract:** Learning robotic manipulation through reinforcement learning (RL) using only
sparse reward signals is still considered a largely unsolved problem.
Leveraging human demonstrations can make the learning...

### [5] Lifelong Robotic Reinforcement Learning by Retaining Experiences
- **ID:** `31059`  
- **Similitud:** 57.9%  
- **Abstract:** Multi-task learning ideally allows robots to acquire a diverse repertoire of
useful skills. However, many multi-task reinforcement learning efforts assume
the robot can collect data from all tasks at ...

## Consulta
> Recent advances in diffusion models for image generation.

## Respuesta Generada
Los avances recientes en modelos de difusión para la generación de imágenes incluyen los siguientes enfoques:

*   **Eficiencia en el muestreo:** Se han desarrollado los *Denoising Diffusion Implicit Models* (DDIMs), una clase de modelos probabilísticos implícitos iterativos que aceleran el proceso de muestreo en comparación con los DDPMs tradicionales, manteniendo el mismo procedimiento de entrenamiento [2].
*   **Ajuste de parámetros:** Se ha introducido un esquema de aprendizaje versátil que permite ajustar paso a paso los parámetros de ruido, optimizando el rendimiento de los modelos de difusión con un número reducido de pasos de eliminación de ruido (*denoising steps*) [4].
*   **Generación condicional:** 
    *   Se ha propuesto *Iterative Latent Variable Refinement* (ILVR), un método que guía el proceso generativo de los DDPMs para obtener imágenes con semántica específica, superando las dificultades causadas por la naturaleza estocástica del proceso original [5].
    *   Se ha presentado *Diffusion-Decoding models with Contrastive representations* (D2C), un paradigma que utiliza un prior basado en difusión para entrenar autoencoders variacionales incondicionales destinados a la generación condicional con pocos datos (*few-shot*) [3].

---
## Evidencias Recuperadas

### [1] Instance Selection for GANs
- **ID:** `15349`  
- **Similitud:** 46.2%  
- **Abstract:** Recent advances in Generative Adversarial Networks (GANs) have led to their
widespread adoption for the purposes of generating high quality synthetic
imagery. While capable of generating photo-realist...

### [2] Denoising Diffusion Implicit Models
- **ID:** `4221`  
- **Similitud:** 47.0%  
- **Abstract:** Denoising diffusion probabilistic models (DDPMs) have achieved high quality
image generation without adversarial training, yet they require simulating a
Markov chain for many steps to produce a sample...

### [3] D2C: Diffusion-Denoising Models for Few-shot Conditional Generation
- **ID:** `3992`  
- **Similitud:** 46.9%  
- **Abstract:** Conditional generative models of high-dimensional images have many
applications, but supervision signals from conditions to images can be
expensive to acquire. This paper describes Diffusion-Decoding ...

### [4] Noise Estimation for Generative Diffusion Models
- **ID:** `3893`  
- **Similitud:** 47.7%  
- **Abstract:** Generative diffusion models have emerged as leading models in speech and
image generation. However, in order to perform well with a small number of
denoising steps, a costly tuning of the set of noise...

### [5] ILVR: Conditioning Method for Denoising Diffusion Probabilistic Models
- **ID:** `3887`  
- **Similitud:** 49.7%  
- **Abstract:** Denoising diffusion probabilistic models (DDPM) have shown remarkable
performance in unconditional image generation. However, due to the
stochasticity of the generative process in DDPM, it is challeng...

# Sección G: Interfaz Web Conversacional

El sistema cuenta con una interfaz web tipo **chat conversacional** desarrollada en **Next.js** (TypeScript), con diseño minimalista monocromático (negro absoluto / blanco).

## Características de la interfaz
- 💬 **Chat continuo**: historial de conversación persistente por sesión
- 📌 **Panel de evidencias**: cada respuesta del asistente incluye un acordeón colapsable con los papers recuperados (título, ID, similitud)
- ⚡ **Efecto de streaming simulado**: la respuesta aparece progresivamente
- 🗂️ **Sidebar de sesiones**: múltiples conversaciones independientes
- 🔴 **Stop generation**: posibilidad de detener la generación en curso

## Arquitectura
```
Usuario (Navegador)
      │
      ▼
Next.js Frontend (AWS Amplify)
      │  POST /api/chat  {query, history}
      ▼
FastAPI Backend (EC2 t2.micro)
      │
      ├── Embedding Model (sentence-transformers)
      ├── FAISS Index (búsqueda vectorial)
      ├── Cross-Encoder (re-ranking)
      └── Google Gemini API (generación)
```

## URL del sistema

| Componente | URL |
|------------|-----|
| Frontend (Chat) | `https://main.d2m68i0enlppbq.amplifyapp.com/` |
| Docs API | `http://98.93.250.131:8000/docs` |

In [20]:
# Verificar conectividad con el backend desplegado
import urllib.request
import json
import ssl
import os

BACKEND_URL = os.environ.get("BACKEND_URL", "https://mustang-region-arc-villages.trycloudflare.com")

def test_backend_connection(backend_url: str) -> None:
    """Verifica que el backend RAG esté activo y responda correctamente."""
    test_payload = json.dumps({
        "query":   "What is deep learning?",
        "history": []
    }).encode('utf-8')
    
    try:
        # Crear un contexto SSL que no valide certificados de forma explícita
        context_ssl = ssl._create_unverified_context()
        
        req = urllib.request.Request(
            f"{backend_url}/api/chat",
            data=test_payload,
            headers={"Content-Type": "application/json"},
            method="POST"
        )
        # Pasamos context_ssl como argumento aquí:
        with urllib.request.urlopen(req, timeout=15, context=context_ssl) as resp:
            data = json.loads(resp.read().decode())
        print(f"✅ Backend activo en: {backend_url}")
        print(f"   Respuesta (primeras 200 chars): {data.get('answer', '')[:200]}")
        print(f"   Evidencias retornadas: {len(data.get('evidences', []))}")
    except Exception as e:
        print(f"⚠️ No se pudo conectar al backend ({backend_url}): {e}")
        print("   Asegúrate de que el backend esté corriendo antes de esta prueba.")

test_backend_connection(BACKEND_URL)

✅ Backend activo en: https://mustang-region-arc-villages.trycloudflare.com
   Respuesta (primeras 200 chars): El contexto proporcionado no ofrece una definición única o explícita de lo que es el *deep learning*, aunque lo describe como una técnica poderosa de aprendizaje de representación [1]. Los documentos 
   Evidencias retornadas: 5


# Sección H: Despliegue en la Nube

## Arquitectura de Despliegue

### Backend: AWS EC2 (t2.micro — Free Tier)

### Frontend: AWS Amplify


## Endpoints disponibles

| Método | Endpoint | Descripción |
|--------|----------|-------------|
| `POST` | `/api/chat` | Pipeline RAG completo (embeddings → FAISS → re-rank → Gemini) |
| `POST` | `/api/feedback` | Guardar feedback de relevancia del usuario |
| `GET`  | `/health` | Health-check del servidor |
| `GET`  | `/docs` | Documentación interactiva (Swagger UI) |

# Sección I: Evaluación del Sistema y de la Generación

## Metodología

Se evalúa el sistema de forma **subjetiva** sobre 5 consultas de prueba, analizando:

| Criterio | Descripción |
|----------|-------------|
| **Corrección** | ¿La respuesta es factualmente correcta? |
| **Relevancia** | ¿Responde directamente a lo que se preguntó? |
| **Fidelidad** | ¿Las afirmaciones están respaldadas por los documentos recuperados? |
| **Integración** | ¿Combina información de múltiples papers de forma coherente? |
| **Reconocimiento de límites** | ¿Reconoce cuándo el corpus no tiene información suficiente? |

Escala: **1** (muy malo) → **5** (excelente)

In [21]:
# ── Definición de consultas de evaluación ────────────────────────────────────────
eval_queries = [
    "What are the main applications of Graph Neural Networks?",
    "How is reinforcement learning used in robotics?",
    "Recent advances in diffusion models for image generation.",
    "Techniques for improving retrieval-augmented generation systems.",
    "What are the best practices for federated learning with privacy constraints?"
]

print("Ejecutando evaluación subjetiva del sistema RAG...")
print("=" * 70)

eval_results = []
for i, query in enumerate(eval_queries):
    print(f"\n[{i+1}/{len(eval_queries)}] '{query[:55]}...'")
    initial  = search_initial(query, k=TOP_K_INITIAL)
    reranked = rerank_results(query, initial, top_k=TOP_K_RERANK)
    result   = generate_rag_answer(query, reranked)
    
    eval_results.append({
        "query":     query,
        "answer":    result["answer"],
        "n_docs":    len(result["evidences"]),
        "evidences": result["evidences"]
    })
    
    # Mostrar top-3 evidencias recuperadas
    print(f"   Documentos recuperados: {len(result['evidences'])}")
    for j, ev in enumerate(result['evidences'][:3]):
        print(f"   [{j+1}] {ev['title'][:60]} (sim: {ev['similarity']:.3f})")

print("\nEvaluación completada.")

Ejecutando evaluación subjetiva del sistema RAG...

[1/5] 'What are the main applications of Graph Neural Networks...'
   Documentos recuperados: 5
   [1] Graph Capsule Convolutional Neural Networks (sim: 0.578)
   [2] AutoGraph: Automated Graph Neural Network (sim: 0.570)
   [3] AutoGraph: Automated Graph Neural Network (sim: 0.570)

[2/5] 'How is reinforcement learning used in robotics?...'
   Documentos recuperados: 5
   [1] Using Deep Reinforcement Learning for the Continuous Control (sim: 0.641)
   [2] Reinforcement Learning for Robotics and Control with Active  (sim: 0.579)
   [3] Setting up a Reinforcement Learning Task with a Real-World R (sim: 0.615)

[3/5] 'Recent advances in diffusion models for image generatio...'
   Documentos recuperados: 5
   [1] Instance Selection for GANs (sim: 0.462)
   [2] Denoising Diffusion Implicit Models (sim: 0.470)
   [3] D2C: Diffusion-Denoising Models for Few-shot Conditional Gen (sim: 0.469)

[4/5] 'Techniques for improving retrieval-augment

In [22]:
# ── Tabla de evaluación subjetiva ────────────────────────────────────────────────
# Scores asignados manualmente tras revisar las respuestas generadas
# Escala: 1=Muy malo, 2=Malo, 3=Regular, 4=Bueno, 5=Excelente

subjective_scores = [
    # query_short,         correcta, relevancia, fidelidad, integracion, limites, observacion
    ("Graph Neural Networks",     5,   5,   5,   4,   3, "Excelente. Cita múltiples papers correctamente."),
    ("RL in Robotics",            4,   5,   4,   4,   3, "Buena cobertura, algunos papers no son del todo relevantes."),
    ("Diffusion Models (images)", 5,   5,   5,   5,   4, "Muy buena integración de papers de difusión."),
    ("RAG Improvements",          4,   4,   4,   3,   4, "El corpus tiene pocos papers sobre RAG explícito."),
    ("Federated Learning Privacy",3,   3,   3,   2,   5, "El corpus tiene cobertura limitada sobre FL+privacy."),
]

print("\n📊 TABLA DE EVALUACIÓN SUBJETIVA")
print("-" * 120)
print(f"{'Consulta':<30} {'Corrección':>10} {'Relevancia':>10} {'Fidelidad':>10} {'Integración':>12} {'Límites':>8} {'Promedio':>9}")
print("-" * 120)

all_avgs = []
for row in subjective_scores:
    name, corr, rel, fid, integ, lim, obs = row
    avg = (corr + rel + fid + integ + lim) / 5
    all_avgs.append(avg)
    print(f"{name:<30} {corr:>10} {rel:>10} {fid:>10} {integ:>12} {lim:>8} {avg:>9.2f}")

print("-" * 120)
global_avg = np.mean(all_avgs)
print(f"{'PROMEDIO GLOBAL':<30} {'':>10} {'':>10} {'':>10} {'':>12} {'':>8} {global_avg:>9.2f}")
print()
print("📝 Observaciones por consulta:")
for row in subjective_scores:
    name, *_, obs = row
    print(f"  • {name}: {obs}")


📊 TABLA DE EVALUACIÓN SUBJETIVA
------------------------------------------------------------------------------------------------------------------------
Consulta                       Corrección Relevancia  Fidelidad  Integración  Límites  Promedio
------------------------------------------------------------------------------------------------------------------------
Graph Neural Networks                   5          5          5            4        3      4.40
RL in Robotics                          4          5          4            4        3      4.00
Diffusion Models (images)               5          5          5            5        4      4.80
RAG Improvements                        4          4          4            3        4      3.80
Federated Learning Privacy              3          3          3            2        5      3.20
------------------------------------------------------------------------------------------------------------------------
PROMEDIO GLOBAL             

In [23]:
# ── Test: consulta fuera del alcance del corpus ──────────────────────────────────
out_of_scope_query = "What are the regulations for AI in healthcare in Ecuador?"
print(f"Consulta fuera del corpus:\n> '{out_of_scope_query}'\n")

initial  = search_initial(out_of_scope_query, k=TOP_K_INITIAL)
reranked = rerank_results(out_of_scope_query, initial, top_k=5)
result   = generate_rag_answer(out_of_scope_query, reranked)

print("Respuesta del sistema RAG:")
print("-" * 60)
print(result['answer'])
print()
print("Documentos recuperados (baja relevancia esperada):")
for ev in result['evidences'][:3]:
    print(f"  • {ev['title'][:70]} (sim: {ev['similarity']:.3f})")

Consulta fuera del corpus:
> 'What are the regulations for AI in healthcare in Ecuador?'

Respuesta del sistema RAG:
------------------------------------------------------------
El contexto proporcionado no contiene información sobre las regulaciones para la inteligencia artificial en el sector salud en Ecuador.

Documentos recuperados (baja relevancia esperada):
  • Explainable AI meets Healthcare: A Study on Heart Disease Dataset (sim: 0.464)
  • Bridging the gap between AI and Healthcare sides: towards developing c (sim: 0.445)
  • A Canonical Architecture For Predictive Analytics on Longitudinal Pati (sim: 0.468)
